# Hebrew CRNN — CTC word-level character aligner (training)

Trains a CRNN (CNN + BiLSTM + CTC) on **word-level crops** from the SCE dataset.
Training on word crops (not full lines) matches the inference scenario exactly.

**Setup:**
1. Create and upload the SCE dataset zip (run locally):
   ```
   cd /mnt/ssd2/cyttic/datasets/sce_dataset && \
   zip -r ~/Dataset_Output.zip Dataset_Output/ \
       --exclude "Dataset_Output/All_Temeplates_Images/*" -x "*.DS_Store"
   ```
2. Upload `Dataset_Output.zip` to Google Drive, share **"Anyone with the link"**,
   paste the link into `DATASET_URL` in Cell 1.
3. Kaggle → Settings → **Internet ON**, **Accelerator GPU T4 x2**.
4. Run All. Download `crnn.pth` from the output panel.

In [ ]:
# ── 0. download SCE dataset from Google Drive ───────────────────────────
!pip install -q -U gdown
import gdown, re, zipfile, os

DATASET_URL = 'https://drive.google.com/file/d/1RdmiH_v4My18XNFNt-gIvmKyE94cbSSy/view?usp=sharing'

m = re.search(r'/d/([\w-]+)|[?&]id=([\w-]+)', DATASET_URL)
FILE_ID = next(g for g in m.groups() if g)
gdown.download(f'https://drive.google.com/uc?id={FILE_ID}',
               '/kaggle/working/Dataset_Output.zip', quiet=False)

print('Extracting …')
with zipfile.ZipFile('/kaggle/working/Dataset_Output.zip') as z:
    z.extractall('/kaggle/working/')

DATASET = '/kaggle/working/Dataset_Output'
assert os.path.isdir(DATASET), 'Extraction failed — check DATASET_URL'
print('Contents:', os.listdir(DATASET))

In [ ]:
# ── 1. imports & constants ──────────────────────────────────────────────
import json, random
from collections import Counter, defaultdict
from pathlib import Path

import cv2
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset

ALPHABET    = list('אבגדהוזחטיכלמנסעפצקרשת') + list('ךםןףץ')
BLANK       = 0
CHAR2IDX    = {c: i + 1 for i, c in enumerate(ALPHABET)}
IDX2CHAR    = {i + 1: c for i, c in enumerate(ALPHABET)}
NUM_CLASSES = len(ALPHABET)   # 27

IMG_H    = 64
MIN_W    = 32
MAX_W    = 640
HIDDEN   = 256
EPOCHS   = 40
BATCH    = 32
LR       = 1e-3
VAL_SPLIT = 0.1
SEED     = 42
OUT      = '/kaggle/working'

random.seed(SEED)
torch.manual_seed(SEED)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device, '| classes:', NUM_CLASSES)

In [ ]:
# ── 2. CRNN model (CNN backbone + LOCAL 1D-conv head) ───────────────────
# A BiLSTM head gives great recognition but USELESS localization: with full
# sequence context, CTC dumps all characters bunched at one edge of the image
# (it only needs the sequence right, not the positions). For character
# localization we use a stack of 1D convs with a small receptive field, so each
# output timestep sees only a local image patch and fires where the letter is.

class CRNN(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES, hidden=HIDDEN):
        super().__init__()
        self.cnn = nn.Sequential(
            # Block 1 — H/2, W/2
            nn.Conv2d(1, 64, 3, padding=1),
            nn.BatchNorm2d(64), nn.ReLU(True),
            nn.MaxPool2d(2, 2),
            # Block 2 — H/4  (height-only pool, width stays W/2)
            nn.Conv2d(64, 128, 3, padding=1),
            nn.BatchNorm2d(128), nn.ReLU(True),
            nn.MaxPool2d((2, 1)),
            # Block 3 — H/8
            nn.Conv2d(128, 256, 3, padding=1),
            nn.BatchNorm2d(256), nn.ReLU(True),
            nn.MaxPool2d((2, 1)),
            # Block 4 — H/16
            nn.Conv2d(256, 256, 3, padding=1),
            nn.BatchNorm2d(256), nn.ReLU(True),
            nn.MaxPool2d((2, 1)),
            nn.AdaptiveAvgPool2d((1, None)),
        )
        # Local-context head (kernel 5 → small receptive field → spatial spikes)
        self.head = nn.Sequential(
            nn.Conv1d(256, hidden, kernel_size=5, padding=2),
            nn.BatchNorm1d(hidden), nn.ReLU(True),
            nn.Conv1d(hidden, hidden, kernel_size=5, padding=2),
            nn.BatchNorm1d(hidden), nn.ReLU(True),
        )
        self.fc = nn.Conv1d(hidden, num_classes + 1, kernel_size=1)

    def forward(self, x):
        feat = self.cnn(x)            # (B, 256, 1, W')
        feat = feat.squeeze(2)        # (B, 256, W')
        feat = self.head(feat)        # (B, hidden, W')
        feat = self.fc(feat)          # (B, C, W')
        feat = feat.permute(2, 0, 1)  # (W', B, C)
        return feat.log_softmax(2)


def encode(text):
    return [CHAR2IDX[c] for c in text if c in CHAR2IDX]


def decode(indices):
    out, prev = [], None
    for i in indices:
        i = int(i)
        if i != BLANK and i != prev:
            out.append(IDX2CHAR.get(i, '?'))
        prev = i
    return ''.join(out)


model = CRNN().to(device)
print('CRNN params:', sum(p.numel() for p in model.parameters()) // 1000, 'K')

In [ ]:
# ── 3. build training data (pre-crop TIFF words to disk ONCE) ───────────
# Loading a full multi-MB TIFF page for every word crop starves the GPU.
# Instead we decode each page exactly once and save its word crops as small
# PNGs, so training reads tiny files and the dataloader keeps the GPU fed.
from collections import defaultdict
from PIL import Image

def build_class_map(json_dir):
    class_texts = defaultdict(Counter)
    for jf in Path(json_dir).glob('*.json'):
        try:
            data = json.load(open(jf, encoding='utf-8'))
        except Exception:
            continue
        for shape in data.get('shapes', []):
            if shape.get('type') != 'word': continue
            cls = shape.get('word_class')
            txt = shape.get('transcript', '').strip()
            if cls is not None and txt:
                class_texts[cls][txt] += 1
    return {cls: ctr.most_common(1)[0][0] for cls, ctr in class_texts.items()}


base      = Path(DATASET)
json_dir  = base / 'Data' / 'json_labels'
img_dir   = base / 'Data' / 'Images'
crop_dir  = Path('/kaggle/working/tiff_crops')
crop_dir.mkdir(exist_ok=True)

print('Building class map …')
class_map = build_class_map(json_dir)
print(f'  {len(class_map)} word classes resolved')

# — source 1: pre-cropped Words_Dataset JPGs (already small) —
jpg_samples = []
for cls_dir in (base / 'Words_Dataset').iterdir():
    if not cls_dir.name.isdigit(): continue
    text = ''.join(c for c in class_map.get(int(cls_dir.name), '') if c in CHAR2IDX)
    if not text: continue
    for p in cls_dir.glob('*.jpg'):
        jpg_samples.append((str(p), text))
print(f'  Words_Dataset JPGs: {len(jpg_samples)}')

# — source 2: extract word crops from each TIFF page exactly once —
tiff_samples = []
json_files = sorted(json_dir.glob('*.json'))
for n_done, jf in enumerate(json_files):
    tif = img_dir / f'{jf.stem}.tif'
    if not tif.exists(): tif = img_dir / f'{jf.stem}.tiff'
    if not tif.exists(): continue
    try:
        data = json.load(open(jf, encoding='utf-8'))
    except Exception:
        continue
    words = [s for s in data.get('shapes', []) if s.get('type') == 'word'
             and ''.join(c for c in s.get('transcript', '') if c in CHAR2IDX)]
    if not words:
        continue
    # decode the full page a SINGLE time (PIL handles old-JPEG TIFFs)
    try:
        page = np.array(Image.open(str(tif)).convert('L'))
    except Exception:
        continue
    for wi, shape in enumerate(words):
        txt = ''.join(c for c in shape['transcript'] if c in CHAR2IDX)
        pts = np.array(shape['points'])
        x0, y0 = int(pts[:, 0].min()), int(pts[:, 1].min())
        x1, y1 = int(pts[:, 0].max()), int(pts[:, 1].max())
        if x1 <= x0 or y1 <= y0:
            continue
        crop = page[y0:y1, x0:x1]
        if crop.size == 0:
            continue
        out = crop_dir / f'{jf.stem}_{wi}.png'
        cv2.imwrite(str(out), crop)
        tiff_samples.append((str(out), txt))
    if (n_done + 1) % 50 == 0:
        print(f'  extracted from {n_done + 1}/{len(json_files)} pages …')

all_samples = jpg_samples + tiff_samples
random.shuffle(all_samples)
print(f'JPG: {len(jpg_samples)}  TIFF crops: {len(tiff_samples)}  total: {len(all_samples)}')

In [ ]:
# ── 4. dataset & dataloader ─────────────────────────────────────────────
# all_samples are now uniformly (path, text) pointing to small PNG/JPG files.

def resize_to_h(img, h=IMG_H):
    oh, ow = img.shape[:2]
    new_w = max(MIN_W, min(MAX_W, int(round(ow * h / oh))))
    return cv2.resize(img, (new_w, h), interpolation=cv2.INTER_AREA)

def augment(img):
    alpha = random.uniform(0.8, 1.2); beta = random.uniform(-20, 20)
    img = np.clip(img.astype(np.float32) * alpha + beta, 0, 255).astype(np.uint8)
    if random.random() < 0.5:
        noise = np.random.normal(0, random.uniform(2, 8), img.shape).astype(np.float32)
        img = np.clip(img.astype(np.float32) + noise, 0, 255).astype(np.uint8)
    if random.random() < 0.5:
        scale = random.uniform(0.85, 1.15)
        new_w = max(MIN_W, min(MAX_W, int(img.shape[1] * scale)))
        img = cv2.resize(img, (new_w, img.shape[0]), interpolation=cv2.INTER_LINEAR)
    if random.random() < 0.3:
        k = np.ones((2, 2), np.uint8)
        img = cv2.dilate(img, k) if random.random() < 0.5 else cv2.erode(img, k)
    return img

class HebrewWordDataset(Dataset):
    def __init__(self, samples, train=True):
        self.samples = samples; self.train = train
    def __len__(self): return len(self.samples)
    def __getitem__(self, idx):
        path, text = self.samples[idx]
        img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
        if img is None or img.size == 0:
            img = np.full((IMG_H, MIN_W), 255, np.uint8); text = text[:1] or 'א'
        img = resize_to_h(img)
        if self.train: img = augment(img)
        labels = [CHAR2IDX[c] for c in text[::-1] if c in CHAR2IDX] or [1]
        return torch.from_numpy(img).float().div_(255).unsqueeze(0), labels, img.shape[1]

def collate_fn(batch):
    imgs, labels_list, widths = zip(*batch)
    max_w = max(img.shape[2] for img in imgs)
    padded = torch.full((len(imgs), 1, IMG_H, max_w), 1.0)
    for i, img in enumerate(imgs): padded[i, :, :, :img.shape[2]] = img
    return (padded,
            torch.tensor([l for ls in labels_list for l in ls], dtype=torch.long),
            torch.tensor([w // 2 for w in widths], dtype=torch.long),
            torch.tensor([len(ls) for ls in labels_list], dtype=torch.long))

n_val = max(1, int(len(all_samples) * VAL_SPLIT))
train_ds = HebrewWordDataset(all_samples[:-n_val], train=True)
val_ds   = HebrewWordDataset(all_samples[-n_val:], train=False)
train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True,  collate_fn=collate_fn,
                          num_workers=4, pin_memory=True, persistent_workers=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH, shuffle=False, collate_fn=collate_fn,
                          num_workers=2, pin_memory=True, persistent_workers=True)
print(f'train: {len(train_ds)}  val: {len(val_ds)}')

In [ ]:
# ── 4b. training loop ───────────────────────────────────────────────────

def edit_distance(a, b):
    m, n = len(a), len(b)
    dp = list(range(n + 1))
    for i in range(1, m + 1):
        prev, dp[0] = dp[0], i
        for j in range(1, n + 1):
            tmp = dp[j]
            dp[j] = prev if a[i-1] == b[j-1] else 1 + min(prev, dp[j], dp[j-1])
            prev = tmp
    return dp[n]


def run_epoch(loader, train):
    model.train(train)
    total_loss = total_ed = total_len = n = 0
    with torch.set_grad_enabled(train):
        for imgs, targets, input_lengths, target_lengths in loader:
            imgs    = imgs.to(device)
            targets = targets.to(device)
            log_probs = model(imgs)        # (T, B, C)
            T = log_probs.shape[0]
            ilens = input_lengths.clamp(max=T).to(device)
            loss  = criterion(log_probs, targets, ilens, target_lengths)
            if train:
                optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 5.0)
                optimizer.step()
            B = imgs.shape[0]
            total_loss += loss.item() * B
            n          += B
            preds_idx = log_probs.argmax(2).permute(1, 0)  # (B, T)
            offset = 0
            for i, tlen in enumerate(target_lengths.tolist()):
                gt   = ''.join(IDX2CHAR.get(x, '?')
                               for x in targets[offset:offset+tlen].tolist())[::-1]
                pred = decode(preds_idx[i].tolist())[::-1]
                total_ed  += edit_distance(pred, gt)
                total_len += max(len(gt), 1)
                offset    += tlen
    return total_loss / max(n, 1), total_ed / max(total_len, 1)


optimizer = torch.optim.Adam(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
criterion = nn.CTCLoss(blank=BLANK, reduction='mean', zero_infinity=True)

best_cer = float('inf')
for epoch in range(1, EPOCHS + 1):
    tr_loss, tr_cer = run_epoch(train_loader, train=True)
    vl_loss, vl_cer = run_epoch(val_loader,   train=False)
    scheduler.step()
    print(f'epoch {epoch:3d}/{EPOCHS}  '
          f'train loss={tr_loss:.4f} CER={tr_cer:.3f}  '
          f'val loss={vl_loss:.4f} CER={vl_cer:.3f}')
    if vl_cer < best_cer:
        best_cer = vl_cer
        torch.save({
            'model_state': model.state_dict(),
            'num_classes':  NUM_CLASSES,
            'hidden':       HIDDEN,
            'epoch':        epoch,
            'val_cer':      vl_cer,
        }, f'{OUT}/crnn.pth')
        print(f'  ✓ saved  best val CER={best_cer:.3f}')

print('Training done. Best val CER:', best_cer)

In [ ]:
# ── 5. sanity check — show a few val predictions ────────────────────────
import matplotlib.pyplot as plt

model.eval()
shown = 0
fig, axes = plt.subplots(2, 4, figsize=(20, 5))

for imgs, targets, input_lengths, target_lengths in val_loader:
    log_probs = model(imgs.to(device))
    preds_idx = log_probs.argmax(2).permute(1, 0)
    offset = 0
    for i, tlen in enumerate(target_lengths.tolist()):
        if shown >= 8: break
        gt   = ''.join(IDX2CHAR.get(x, '?')
                       for x in targets[offset:offset+tlen].tolist())[::-1]
        pred = decode(preds_idx[i].tolist())[::-1]
        ax   = axes[shown // 4][shown % 4]
        ax.imshow(imgs[i, 0].numpy(), cmap='gray', aspect='auto')
        ok = pred == gt
        ax.set_title(f'GT:   {gt[:30]}\nPred: {pred[:30]}',
                     fontsize=7, color='green' if ok else 'red')
        ax.axis('off')
        shown  += 1
        offset += tlen
    if shown >= 8: break

plt.tight_layout()
plt.savefig(f'{OUT}/crnn_samples.png', dpi=100)
plt.show()
print('Saved files:', [f for f in os.listdir(OUT) if f.endswith(('.pth', '.png'))])

In [ ]:
# ── 6. get a direct download link for crnn.pth ──────────────────────────
import requests
r = requests.post('https://pixeldrain.com/api/file/crnn.pth',
                  files={'file': open('/kaggle/working/crnn.pth', 'rb')})
print('https://pixeldrain.com/u/' + r.json()['id'])